# Setup
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv

dotenv.load_dotenv()
ENV_PG_CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

In [ ]:
from langchain_postgres import PGEngine
from sqlalchemy.ext.asyncio import create_async_engine

# エンジン初期化
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

sa_engine = create_async_engine(ENV_PG_CONNECTION_STRING, pool_size=5)
pg_engine = PGEngine.from_engine(sa_engine)

# Extract

In [ ]:
from pathlib import Path
from assistant_agent.loaders import ObsidianLoader

# Vault から読み込み
vault_path = Path("../../docs/dataset_obsidian/")
loader = ObsidianLoader(vault_path)
docs = loader.load()

print(f"{len(docs)} 件のノートを読み込みました")

# Load

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import SecretStr

from assistant_agent.entities.postgres import ObsidianEntity, ObsidianChunkEntity
from assistant_agent.services import VaultObsidianRetriever
from assistant_agent.store import PostgresStoreConnector

assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

store_conn = PostgresStoreConnector(ENV_PG_CONNECTION_STRING)
sa_engine = store_conn.get_engine()
obsidian_retriever = VaultObsidianRetriever(
    vault_entity=ObsidianEntity,
    chunk_entity=ObsidianChunkEntity,
    store_conn=store_conn,
    splitter=RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=200),
    embed_model=GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        api_key=SecretStr(ENV_GEMINI_API_KEY),
    ),
)

In [ ]:
from sqlalchemy import text
from assistant_agent.entities.base import VaultUtils
from assistant_agent.entities.postgres import AssetBase, AppBase, ObsidianEntity

# DB へ取り込み
async with sa_engine.begin() as conn:
    await conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector"))
    await conn.run_sync(AssetBase.metadata.create_all)
    await conn.run_sync(AppBase.metadata.create_all)
await VaultUtils.sync_docs(docs, sa_engine, ObsidianEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
await obsidian_retriever.sync_chunks()

# Retrieval

In [ ]:
chunks = await obsidian_retriever._vector_store.asimilarity_search_with_score(
    "プロンプトエンジニアリング",
    k=5,
    filter={"file_path": {"$like": "03_Structure/%"}},
)
for item, score in chunks:
    print(item.page_content)
    print("==================")